In [ ]:
import os
from glob import glob
import geopandas
import pandas
import fiona
import numpy
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import rasterio
# from analysis_utils import *

In [ ]:
# In Europe 40% of woodlands are intermingled with natural / semi-natural non-forested lands, agriculture and artificial lands in 1km2 surroundings

# In EU, forest edges are mainly (60%) along intensive land uses

In [ ]:
base_path = 'Z:\\jamaica\\Inputs'

In [ ]:
output_path = 'Z:\\jamaica\\Results'

In [ ]:
test_output_path = 'D:\\test_jamaica'

In [ ]:
intermediate_results = os.path.join(output_path,"Buffer_intersections")
if os.path.exists(intermediate_results) is False:
    os.mkdir(intermediate_results)

In [ ]:
jamaica_crs = 3448

In [ ]:
jamaicaboundary = geopandas.read_file(os.path.join(base_path, 'jamaica.gpkg'))
jamaicaboundary = jamaicaboundary.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
jamaicaboundary['area_hectares'] = 0.0001*jamaicaboundary.geometry.area # Convert area to hectares
jamaica_total_area = jamaicaboundary['area_hectares'].sum()
#jamaicaboundary['area'] = jamaicaboundary.geometry.area 
#jamaica_total_area = jamaicaboundary['area'].sum()

In [ ]:
jamaicaboundary

In [ ]:
landcover = geopandas.read_file(os.path.join(base_path, '2013_landuse_landcover.gpkg'))[["OBJECTID","geometry","Classify"]]
landcover = landcover.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
#landcover


## FORESTS

### Primary Forest

In [ ]:
primary_forest_classes = ["Closed broadleaved forest (Primary Forest)"]

In [ ]:
#secondary_forest = geopandas.read_file(os.path.join(base_path, 'Secondary_Forest.gpkg'))[["OBJECTID","geometry"]]
primary_forest = landcover[landcover["Classify"].isin(primary_forest_classes)]
primary_forest = primary_forest.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
primary_forest["total_area"]= 0.0001*primary_forest.geometry.area #convert to hectares
#secondary_forest_copy = secondary_forest.copy()
#secondary_forest_copy.rename(columns={"OBJECTID":"forest_id"},inplace=True)

In [ ]:
for distance in [5, 100, 250, 500, 1000]: 
    primary_forest_copy = primary_forest.copy()
    primary_forest_copy.rename(columns={"OBJECTID":"forest_id"},inplace=True)
    primary_forest_copy["geometry"] = primary_forest_copy.geometry.buffer(distance=distance) #dry_forest buffered different distances
    primary_forest_copy = primary_forest_copy[["forest_id","geometry"]] \
    .overlay(primary_forest[["OBJECTID","geometry"]].set_geometry("geometry"), how='difference') 
    primary_forest_surrounding_landcover_only = primary_forest_copy[["forest_id","geometry"]] \
    .overlay(landcover[~landcover["Classify"].isin(primary_forest_classes)].set_geometry("geometry"), how='intersection') #intersection of dry_forest and surrounding landcover to see what neighbours the dry_forest
    #secondary_forest_surrounding_landcover_only = geopandas.GeoDataFrame(secondary_forest_and_surroundings, geometry="geometry",crs=f"EPSG:{jamaica_crs}") #create a dataframe of the intersected primary forest and surrounding landcover
    #intersection with landcover
    # secondary_forest_and_surroundings.to_file(os.path.join(output_path, f'secondary_forest_and_surroundings_{distance}m_buffer.gpkg'),driver="GPKG") #save primary forest and intersected surrounding landcover to a geopackage for viewing in QGIS
    #print (secondary_forest_and_surroundings)
    primary_forest_surrounding_landcover_only = geopandas.GeoDataFrame(primary_forest_surrounding_landcover_only, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
    primary_forest_surrounding_landcover_only["area"]= primary_forest_surrounding_landcover_only.geometry.area
    # secondary_forest_surrounding_landcover_only.to_file(os.path.join(output_path, f'secondary_forest_surrounding_landcover_only_{distance}m_buffer.gpkg'),driver="GPKG")
    df = primary_forest_surrounding_landcover_only
    total_forest_area = df['area'].sum()
    df['percentage'] = df['area']/df.groupby(['forest_id'])['area'].transform('sum')
    df = df.groupby(['Classify'])['area'].sum().reset_index()
    df['area_percentage'] = df['area']/total_forest_area
    #print (df)
    
    df.to_csv(os.path.join(output_path, f'primary_forest_surrounding_landcover_only_{distance}m_buffer.csv'))


In [ ]:
#primary_forest = geopandas.read_file(os.path.join(base_path, 'Primary_Forest.gpkg'))[["OBJECTID","geometry"]]
#primary_forest = primary_forest.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
#primary_forest["total_area"]= 0.0001*primary_forest.geometry.area #convert to hectares
#primary_forest_copy = primary_forest.copy()
#primary_forest_copy.rename(columns={"OBJECTID":"forest_id"},inplace=True)

In [ ]:
#for distance in [5, 100, 250, 500, 1000]: 
    #primary_forest_copy = primary_forest.copy()
    #primary_forest_copy.rename(columns={"OBJECTID":"forest_id"},inplace=True)
    #primary_forest_copy["geometry"] = primary_forest_copy.geometry.buffer(distance=distance) #dry_forest buffered different distances
    #primary_forest_and_surroundings = primary_forest_copy[["forest_id","geometry"]] \
    #.overlay(landcover.set_geometry("geometry"), how='intersection') #intersection of dry_forest and surrounding landcover to see what neighbours the dry_forest
    #primary_forest_and_surroundings = geopandas.GeoDataFrame(primary_forest_and_surroundings, geometry="geometry",crs=f"EPSG:{jamaica_crs}") #create a dataframe of the intersected primary forest and surrounding landcover
    # primary_forest_and_surroundings.to_file(os.path.join(output_path, f'primary_forest_and_surroundings_{distance}m_buffer.gpkg'),driver="GPKG") #save primary forest and intersected surrounding landcover to a geopackage for viewing in QGIS
   
    #primary_forest_surrounding_landcover_only = primary_forest_and_surroundings[["forest_id","Classify","geometry"]] \
    #.overlay(primary_forest.set_geometry("geometry"), how='difference')
    #primary_forest_surrounding_landcover_only = geopandas.GeoDataFrame(primary_forest_surrounding_landcover_only, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
    #primary_forest_surrounding_landcover_only["area"]= primary_forest_surrounding_landcover_only.geometry.area
    #primary_forest_surrounding_landcover_only.to_file(os.path.join(intermediate_results, f'primary_forest_surrounding_landcover_only_{distance}m_buffer.gpkg'),driver="GPKG")
    #df = primary_forest_surrounding_landcover_only
    #total_forest_area = df['area'].sum()
    #df['percentage'] = df['area']/df.groupby(['forest_id'])['area'].transform('sum')
    #df = df.groupby(['Classify'])['area'].sum().reset_index()
    #df['area_percentage'] = df['area']/total_forest_area
    #print (df)
    
    #df.to_csv(os.path.join(output_path, f'primary_forest_surrounding_landcover_only_{distance}m_buffer.csv'))


### Secondary Forest 

In [ ]:
secondary_forest_classes = ["Bamboo and Secondary Forest",
                            "Disturbed broadleaved forest (Secondary Forest)",
                            "Fields and Secondary Forest",
                            "Fields or Secondary Forest/Pine Plantation",
                            "Secondary Forest"]

In [ ]:
#secondary_forest = geopandas.read_file(os.path.join(base_path, 'Secondary_Forest.gpkg'))[["OBJECTID","geometry"]]
secondary_forest = landcover[landcover["Classify"].isin(secondary_forest_classes)]
secondary_forest = secondary_forest.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
secondary_forest["total_area"]= 0.0001*secondary_forest.geometry.area #convert to hectares
#secondary_forest_copy = secondary_forest.copy()
#secondary_forest_copy.rename(columns={"OBJECTID":"forest_id"},inplace=True)

In [ ]:
for distance in [100, 250, 500, 1000]: 
    secondary_forest_copy = secondary_forest.copy()
    secondary_forest_copy.rename(columns={"OBJECTID":"forest_id"},inplace=True)
    secondary_forest_copy["geometry"] = secondary_forest_copy.geometry.buffer(distance=distance) #dry_forest buffered different distances
    secondary_forest_copy = secondary_forest_copy[["forest_id","geometry"]] \
    .overlay(secondary_forest[["OBJECTID","geometry"]].set_geometry("geometry"), how='difference') 
    secondary_forest_surrounding_landcover_only = secondary_forest_copy[["forest_id","geometry"]] \
    .overlay(landcover[~landcover["Classify"].isin(secondary_forest_classes)].set_geometry("geometry"), how='intersection') #intersection of dry_forest and surrounding landcover to see what neighbours the dry_forest
    #secondary_forest_surrounding_landcover_only = geopandas.GeoDataFrame(secondary_forest_and_surroundings, geometry="geometry",crs=f"EPSG:{jamaica_crs}") #create a dataframe of the intersected primary forest and surrounding landcover
    #intersection with landcover
    # secondary_forest_and_surroundings.to_file(os.path.join(output_path, f'secondary_forest_and_surroundings_{distance}m_buffer.gpkg'),driver="GPKG") #save primary forest and intersected surrounding landcover to a geopackage for viewing in QGIS
    #print (secondary_forest_and_surroundings)
    secondary_forest_surrounding_landcover_only = geopandas.GeoDataFrame(secondary_forest_surrounding_landcover_only, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
    secondary_forest_surrounding_landcover_only["area"]= secondary_forest_surrounding_landcover_only.geometry.area
    # secondary_forest_surrounding_landcover_only.to_file(os.path.join(output_path, f'secondary_forest_surrounding_landcover_only_{distance}m_buffer.gpkg'),driver="GPKG")
    df = secondary_forest_surrounding_landcover_only
    total_forest_area = df['area'].sum()
    df['percentage'] = df['area']/df.groupby(['forest_id'])['area'].transform('sum')
    df = df.groupby(['Classify'])['area'].sum().reset_index()
    df['area_percentage'] = df['area']/total_forest_area
    #print (df)
    
    df.to_csv(os.path.join(test_output_path, f'secondary_forest_surrounding_landcover_only_{distance}m_buffer.csv'))


### Dry Forest

In [ ]:
dry_forest_classes = ["Open dry forest - Short",
                            "Open dry forest - Tall (Woodland/Savanna)",
                            ]

In [ ]:
#secondary_forest = geopandas.read_file(os.path.join(base_path, 'Secondary_Forest.gpkg'))[["OBJECTID","geometry"]]
dry_forest = landcover[landcover["Classify"].isin(dry_forest_classes)]
dry_forest = dry_forest.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
dry_forest["total_area"]= 0.0001*dry_forest.geometry.area #convert to hectares
#secondary_forest_copy = secondary_forest.copy()
#secondary_forest_copy.rename(columns={"OBJECTID":"forest_id"},inplace=True)

In [ ]:
for distance in [5, 100, 250, 500, 1000]: 
    dry_forest_copy = dry_forest.copy()
    dry_forest_copy.rename(columns={"OBJECTID":"forest_id"},inplace=True)
    dry_forest_copy["geometry"] = dry_forest_copy.geometry.buffer(distance=distance) #dry_forest buffered different distances
    dry_forest_copy = dry_forest_copy[["forest_id","geometry"]] \
    .overlay(dry_forest[["OBJECTID","geometry"]].set_geometry("geometry"), how='difference') 
    dry_forest_surrounding_landcover_only = dry_forest_copy[["forest_id","geometry"]] \
    .overlay(landcover[~landcover["Classify"].isin(dry_forest_classes)].set_geometry("geometry"), how='intersection') #intersection of dry_forest and surrounding landcover to see what neighbours the dry_forest
    #secondary_forest_surrounding_landcover_only = geopandas.GeoDataFrame(secondary_forest_and_surroundings, geometry="geometry",crs=f"EPSG:{jamaica_crs}") #create a dataframe of the intersected primary forest and surrounding landcover
    #intersection with landcover
    # secondary_forest_and_surroundings.to_file(os.path.join(output_path, f'secondary_forest_and_surroundings_{distance}m_buffer.gpkg'),driver="GPKG") #save primary forest and intersected surrounding landcover to a geopackage for viewing in QGIS
    #print (secondary_forest_and_surroundings)
    dry_forest_surrounding_landcover_only = geopandas.GeoDataFrame(dry_forest_surrounding_landcover_only, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
    dry_forest_surrounding_landcover_only["area"]= dry_forest_surrounding_landcover_only.geometry.area
    # secondary_forest_surrounding_landcover_only.to_file(os.path.join(output_path, f'secondary_forest_surrounding_landcover_only_{distance}m_buffer.gpkg'),driver="GPKG")
    df = dry_forest_surrounding_landcover_only
    total_forest_area = df['area'].sum()
    df['percentage'] = df['area']/df.groupby(['forest_id'])['area'].transform('sum')
    df = df.groupby(['Classify'])['area'].sum().reset_index()
    df['area_percentage'] = df['area']/total_forest_area
    #print (df)
    
    df.to_csv(os.path.join(output_path, f'dry_forest_surrounding_landcover_only_{distance}m_buffer.csv'))


### Wetlands

In [ ]:
wetland_classes = ["Herbaceous Wetland"]

In [ ]:
#secondary_forest = geopandas.read_file(os.path.join(base_path, 'Secondary_Forest.gpkg'))[["OBJECTID","geometry"]]
wetland = landcover[landcover["Classify"].isin(wetland_classes)]
wetland = wetland.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
wetland["total_area"]= 0.0001*wetland.geometry.area #convert to hectares
#secondary_forest_copy = secondary_forest.copy()
#secondary_forest_copy.rename(columns={"OBJECTID":"forest_id"},inplace=True)

In [ ]:
for distance in [5, 100, 250, 500, 1000]: 
    wetland_copy = wetland.copy()
    wetland_copy.rename(columns={"OBJECTID":"wetland_id"},inplace=True)
    wetland_copy["geometry"] = wetland_copy.geometry.buffer(distance=distance) #dry_forest buffered different distances
    wetland_copy = wetland_copy[["wetland_id","geometry"]] \
    .overlay(wetland[["OBJECTID","geometry"]].set_geometry("geometry"), how='difference') 
    wetland_surrounding_landcover_only = wetland_copy[["wetland_id","geometry"]] \
    .overlay(landcover[~landcover["Classify"].isin(wetland_classes)].set_geometry("geometry"), how='intersection') #intersection of dry_forest and surrounding landcover to see what neighbours the dry_forest
    #secondary_forest_surrounding_landcover_only = geopandas.GeoDataFrame(secondary_forest_and_surroundings, geometry="geometry",crs=f"EPSG:{jamaica_crs}") #create a dataframe of the intersected primary forest and surrounding landcover
    #intersection with landcover
    # secondary_forest_and_surroundings.to_file(os.path.join(output_path, f'secondary_forest_and_surroundings_{distance}m_buffer.gpkg'),driver="GPKG") #save primary forest and intersected surrounding landcover to a geopackage for viewing in QGIS
    #print (secondary_forest_and_surroundings)
    wetland_surrounding_landcover_only = geopandas.GeoDataFrame(wetland_surrounding_landcover_only, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
    wetland_surrounding_landcover_only["area"]= wetland_surrounding_landcover_only.geometry.area
    # secondary_forest_surrounding_landcover_only.to_file(os.path.join(output_path, f'secondary_forest_surrounding_landcover_only_{distance}m_buffer.gpkg'),driver="GPKG")
    df = wetland_surrounding_landcover_only
    total_wetland_area = df['area'].sum()
    df['percentage'] = df['area']/df.groupby(['wetland_id'])['area'].transform('sum')
    df = df.groupby(['Classify'])['area'].sum().reset_index()
    df['area_percentage'] = df['area']/total_wetland_area
    #print (df)
    
    df.to_csv(os.path.join(output_path, f'wetland_surrounding_landcover_only_{distance}m_buffer.csv'))


In [ ]:
wetlands = geopandas.read_file(os.path.join(base_path, 'Wetland.gpkg'))[["OBJECTID","geometry"]]
wetlands = wetlands.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
wetlands["total_area"]= 0.0001*wetlands.geometry.area #convert to hectares
wetlands_copy = wetlands.copy()
wetlands_copy.rename(columns={"OBJECTID":"wetland_id"},inplace=True)

In [ ]:
for distance in [5, 100, 250, 500, 1000]: 
    wetlands_copy["geometry"] = wetlands_copy.geometry.buffer(distance=distance) #wetlands buffered different distances
    wetlands_and_surroundings = wetlands_copy[["wetland_id","geometry"]] \
    .overlay(landcover.set_geometry("geometry"), how='intersection') #intersection of wetlands and surrounding landcover to see what neighbours the dry_forest
    wetlands_and_surroundings = geopandas.GeoDataFrame(wetlands_and_surroundings, geometry="geometry",crs=f"EPSG:{jamaica_crs}") #create a dataframe of the intersected primary forest and surrounding landcover
    # wetlands_and_surroundings.to_file(os.path.join(output_path, f'wetlands_and_surroundings_{distance}m_buffer.gpkg'),driver="GPKG") #save primary forest and intersected surrounding landcover to a geopackage for viewing in QGIS
   
    wetlands_surrounding_landcover_only = wetlands_and_surroundings[["wetland_id","Classify","geometry"]] \
    .overlay(wetlands.set_geometry("geometry"), how='difference')
    wetlands_surrounding_landcover_only = geopandas.GeoDataFrame(wetlands_surrounding_landcover_only, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
    wetlands_surrounding_landcover_only["area"]= wetlands_surrounding_landcover_only.geometry.area
    wetlands_surrounding_landcover_only.to_file(os.path.join(output_path, f'wetlands_surrounding_landcover_only_{distance}m_buffer.gpkg'),driver="GPKG")
    df = wetlands_surrounding_landcover_only
    total_wetlands_area = df['area'].sum()
    df['percentage'] = df['area']/df.groupby(['wetland_id'])['area'].transform('sum')
    df = df.groupby(['Classify'])['area'].sum().reset_index()
    df['area_percentage'] = df['area']/total_wetlands_area
    #print (df)
    
    df.to_csv(os.path.join(output_path, f'wetlands_surrounding_landcover_only_{distance}m_buffer.csv'))


In [ ]:
Plantations

In [ ]:
plantations_classes = ["Plantation: Tree crops, shrub crops, sugar cane, banana",
                            "Hardwood Plantation: Euculytus",
                            "Hardwood Plantation: Mahoe",
                            "Hardwood Plantation: Mahogany",
                            "Hardwood Plantation: Mixed"]

In [ ]:
#secondary_forest = geopandas.read_file(os.path.join(base_path, 'Secondary_Forest.gpkg'))[["OBJECTID","geometry"]]
plantations = landcover[landcover["Classify"].isin(plantations_classes)]
plantations = plantations.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
plantations["total_area"]= 0.0001*plantations.geometry.area #convert to hectares
plantations_copy = plantations.copy()
plantations_copy.rename(columns={"OBJECTID":"plantations_id"},inplace=True)

In [ ]:
for distance in [5, 100, 250, 500, 1000]: 
    plantations_copy["geometry"] = plantations_copy.geometry.buffer(distance=distance) #dry_forest buffered different distances
    plantations_copy = plantations_copy[["plantations_id","geometry"]] \
    .overlay(plantations[["OBJECTID","geometry"]].set_geometry("geometry"), how='difference') 
    plantations_surrounding_landcover_only = plantations_copy[["plantations_id","geometry"]] \
    .overlay(landcover[~landcover["Classify"].isin(plantations_classes)].set_geometry("geometry"), how='intersection') #intersection of dry_forest and surrounding landcover to see what neighbours the dry_forest
    #secondary_forest_surrounding_landcover_only = geopandas.GeoDataFrame(secondary_forest_and_surroundings, geometry="geometry",crs=f"EPSG:{jamaica_crs}") #create a dataframe of the intersected primary forest and surrounding landcover
    #intersection with landcover
    # secondary_forest_and_surroundings.to_file(os.path.join(output_path, f'secondary_forest_and_surroundings_{distance}m_buffer.gpkg'),driver="GPKG") #save primary forest and intersected surrounding landcover to a geopackage for viewing in QGIS
    #print (secondary_forest_and_surroundings)
    plantations_surrounding_landcover_only = geopandas.GeoDataFrame(plantations_surrounding_landcover_only, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
    plantations_surrounding_landcover_only["area"]= plantations_surrounding_landcover_only.geometry.area
    # secondary_forest_surrounding_landcover_only.to_file(os.path.join(output_path, f'secondary_forest_surrounding_landcover_only_{distance}m_buffer.gpkg'),driver="GPKG")
    df = plantations_surrounding_landcover_only
    total_plantations_area = df['area'].sum()
    df['percentage'] = df['area']/df.groupby(['plantations_id'])['area'].transform('sum')
    df = df.groupby(['Classify'])['area'].sum().reset_index()
    df['area_percentage'] = df['area']/total_plantations_area
    #print (df)
    
    df.to_csv(os.path.join(output_path, f'plantations_surrounding_landcover_only_{distance}m_buffer.csv'))


In [ ]:
fields

In [ ]:
fields_classes = ["Bamboo and Fields",
                            "Fields  and Bamboo",
                            "Fields: Bare Land",
                            "Fields: Herbaceous crops, fallow, cultivated vegetables",
                            "Fields: Pasture,Human disturbed, grassland"]

In [ ]:
#fields = geopandas.read_file(os.path.join(base_path, 'Secondary_Forest.gpkg'))[["OBJECTID","geometry"]]
fields = landcover[landcover["Classify"].isin(fields_classes)]
fields = fields.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
fields["total_area"]= 0.0001*fields.geometry.area #convert to hectares
fields_copy = fields.copy()
fields_copy.rename(columns={"OBJECTID":"fields_id"},inplace=True)

In [ ]:
for distance in [5, 100, 250, 500, 1000]: 
    fields_copy["geometry"] = fields_copy.geometry.buffer(distance=distance) #dry_forest buffered different distances
    fields_copy = fields_copy[["fields_id","geometry"]] \
    .overlay(fields[["OBJECTID","geometry"]].set_geometry("geometry"), how='difference') 
    fields_surrounding_landcover_only = fields_copy[["fields_id","geometry"]] \
    .overlay(landcover[~landcover["Classify"].isin(fields_classes)].set_geometry("geometry"), how='intersection') #intersection of dry_forest and surrounding landcover to see what neighbours the dry_forest
    #secondary_forest_surrounding_landcover_only = geopandas.GeoDataFrame(secondary_forest_and_surroundings, geometry="geometry",crs=f"EPSG:{jamaica_crs}") #create a dataframe of the intersected primary forest and surrounding landcover
    #intersection with landcover
    # secondary_forest_and_surroundings.to_file(os.path.join(output_path, f'secondary_forest_and_surroundings_{distance}m_buffer.gpkg'),driver="GPKG") #save primary forest and intersected surrounding landcover to a geopackage for viewing in QGIS
    #print (secondary_forest_and_surroundings)
    fields_surrounding_landcover_only = geopandas.GeoDataFrame(fields_surrounding_landcover_only, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
    fields_surrounding_landcover_only["area"]= fields_surrounding_landcover_only.geometry.area
    # secondary_forest_surrounding_landcover_only.to_file(os.path.join(output_path, f'secondary_forest_surrounding_landcover_only_{distance}m_buffer.gpkg'),driver="GPKG")
    df = fields_surrounding_landcover_only
    total_fields_area = df['area'].sum()
    df['percentage'] = df['area']/df.groupby(['fields_id'])['area'].transform('sum')
    df = df.groupby(['Classify'])['area'].sum().reset_index()
    df['area_percentage'] = df['area']/total_fields_area
    #print (df)
    
    df.to_csv(os.path.join(output_path, f'fields_surrounding_landcover_only_{distance}m_buffer.csv'))


In [ ]:
all_agriculture

In [ ]:
all_agriculture_classes = ["Bamboo and Fields",
                            "Fields  and Bamboo",
                            "Fields: Bare Land",
                            "Fields: Herbaceous crops, fallow, cultivated vegetables",
                            "Fields: Pasture,Human disturbed, grassland"
                            "Hardwood Plantation: Euculytus",
                            "Hardwood Plantation: Mahoe",
                            "Hardwood Plantation: Mahogany",
                            "Hardwood Plantation: Mixed"
                            "Plantation: Tree crops, shrub crops, sugar cane, banana"]

In [ ]:
#fields = geopandas.read_file(os.path.join(base_path, 'Secondary_Forest.gpkg'))[["OBJECTID","geometry"]]
all_agriculture = landcover[landcover["Classify"].isin(all_agriculture_classes)]
all_agriculture = all_agriculture.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
all_agriculture["total_area"]= 0.0001*all_agriculture.geometry.area #convert to hectares
all_agriculture_copy = all_agriculture.copy()
all_agriculture_copy.rename(columns={"OBJECTID":"all_agriculture_id"},inplace=True)

In [ ]:
for distance in [5, 100, 250, 500, 1000]: 
    all_agriculture_copy["geometry"] = all_agriculture_copy.geometry.buffer(distance=distance) #dry_forest buffered different distances
    all_agriculture_copy = all_agriculture_copy[["all_agriculture_id","geometry"]] \
    .overlay(all_agriculture[["OBJECTID","geometry"]].set_geometry("geometry"), how='difference') 
    all_agriculture_surrounding_landcover_only = all_agriculture_copy[["all_agriculture_id","geometry"]] \
    .overlay(landcover[~landcover["Classify"].isin(all_agriculture_classes)].set_geometry("geometry"), how='intersection') #intersection of dry_forest and surrounding landcover to see what neighbours the dry_forest
    #secondary_forest_surrounding_landcover_only = geopandas.GeoDataFrame(secondary_forest_and_surroundings, geometry="geometry",crs=f"EPSG:{jamaica_crs}") #create a dataframe of the intersected primary forest and surrounding landcover
    #intersection with landcover
    # secondary_forest_and_surroundings.to_file(os.path.join(output_path, f'secondary_forest_and_surroundings_{distance}m_buffer.gpkg'),driver="GPKG") #save primary forest and intersected surrounding landcover to a geopackage for viewing in QGIS
    #print (secondary_forest_and_surroundings)
    all_agriculture_surrounding_landcover_only = geopandas.GeoDataFrame(all_agriculture_surrounding_landcover_only, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
    all_agriculture_surrounding_landcover_only["area"]= all_agriculture_surrounding_landcover_only.geometry.area
    # secondary_forest_surrounding_landcover_only.to_file(os.path.join(output_path, f'secondary_forest_surrounding_landcover_only_{distance}m_buffer.gpkg'),driver="GPKG")
    df = all_agriculture_surrounding_landcover_only
    total_all_agriculture_area = df['area'].sum()
    df['percentage'] = df['area']/df.groupby(['all_agriculture_id'])['area'].transform('sum')
    df = df.groupby(['Classify'])['area'].sum().reset_index()
    df['area_percentage'] = df['area']/total_all_agriculture_area
    #print (df)
    
    df.to_csv(os.path.join(output_path, f'all_agriculture_surrounding_landcover_only_{distance}m_buffer.csv'))


### All forest

In [ ]:
all_forest = geopandas.read_file(os.path.join(base_path, 'All_forest.gpkg'))[["OBJECTID","geometry"]]
all_forest = all_forest.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
all_forest["total_area"]= 0.0001*all_forest.geometry.area #convert to hectares
all_forest_copy = all_forest.copy()
all_forest_copy.rename(columns={"OBJECTID":"forest_id"},inplace=True)

In [ ]:
for distance in [5, 100, 250, 500, 1000]: 
    all_forest_copy["geometry"] = all_forest_copy.geometry.buffer(distance=distance) #all_forest buffered different distances
    all_forest_and_surroundings = all_forest_copy[["forest_id","geometry"]] \
    .overlay(landcover.set_geometry("geometry"), how='intersection') #intersection of all_forest and surrounding landcover to see what neighbours the dry_forest
    all_forest_and_surroundings = geopandas.GeoDataFrame(all_forest_and_surroundings, geometry="geometry",crs=f"EPSG:{jamaica_crs}") #create a dataframe of the intersected all_forest and surrounding landcover
    # all_forest_and_surroundings.to_file(os.path.join(output_path, f'all_forest_and_surroundings_{distance}m_buffer.gpkg'),driver="GPKG") #save all_forest and intersected surrounding landcover to a geopackage for viewing in QGIS
   
    all_forest_surrounding_landcover_only = all_forest_and_surroundings[["forest_id","Classify","geometry"]] \
    .overlay(all_forest.set_geometry("geometry"), how='difference')
    all_forest_surrounding_landcover_only = geopandas.GeoDataFrame(all_forest_surrounding_landcover_only, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
    all_forest_surrounding_landcover_only["area"]= all_forest_surrounding_landcover_only.geometry.area
    all_forest_surrounding_landcover_only.to_file(os.path.join(output_path, f'all_forest_surrounding_landcover_only_{distance}m_buffer.gpkg'),driver="GPKG")
    df = all_forest_surrounding_landcover_only
    total_all_forest_area = df['area'].sum()
    df['percentage'] = df['area']/df.groupby(['forest_id'])['area'].transform('sum')
    df = df.groupby(['Classify'])['area'].sum().reset_index()
    df['area_percentage'] = df['area']/total_all_forest_area
    #print (df)
    
    df.to_csv(os.path.join(output_path, f'all_forest_surrounding_landcover_only_{distance}m_buffer.csv'))


### Bauxite

In [ ]:
#read in the files
bauxite = geopandas.read_file(os.path.join(base_path, 'nsmdb-bauxite_reserves.gpkg'))
landcover = geopandas.read_file(os.path.join(base_path, '2013_landuse_landcover.gpkg'))[["OBJECTID","geometry","Classify"]]
all_protected_areas = geopandas.read_file(os.path.join(base_path, 'allprotectedareas.gpkg'))
all_protected_areas = all_protected_areas.rename(columns={"Region":"protected_region"},inplace=True) #renaming relevant columns because when I went to intersect there were two that were similarly named and it was not happy



In [ ]:
bauxite["total_area"]=bauxite.geometry.area.sum()
all_protected_areas.to_file(os.path.join(output_path, f'all_protected_areas.gpkg'),driver="GPKG")
all_protected_areas["total_area"]=all_protected_areas.geometry.area.sum()
landcover_bauxite = landcover \
    .overlay(bauxite.set_geometry("geometry"), how='intersection') #intersecting landcover with bauxite
landcover_bauxite = geopandas.GeoDataFrame(landcover_bauxite, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
landcover_bauxite.to_file(os.path.join(output_path, f'landcover_bauxite.gpkg'),driver="GPKG")
landcover_bauxite["total_area"]=landcover_bauxite.geometry.area.sum()
landcover_bauxite["percentage_of_jamaica"] = 100.0*landcover_bauxite["total_area"]/jamaica_total_area #to find out the percentage of landcover on bauxite that is protected
landcover_bauxite.to_csv(os.path.join(output_path, 'landcover_bauxite.csv'))

landcover_bauxite_classified = landcover_bauxite.groupby('Classify').sum()
landcover_bauxite_allprotected = landcover_bauxite \
    .overlay(all_protected_areas.set_geometry("geometry"), how='intersection') #intersecting landcover, bauxite with protected areas
landcover_bauxite_allprotected = geopandas.GeoDataFrame(landcover_bauxite_allprotected, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
landcover_bauxite_allprotected.to_file(os.path.join(output_path, f'landcover_bauxite_allprotected.gpkg'),driver="GPKG")

landcover_bauxite_allprotected["total_area"] = landcover_bauxite_allprotected.geometry.area.sum()
landcover_bauxite_allprotected["area_percentage"] = 100.0*landcover_bauxite_allprotected["total_area"]/landcover_bauxite["total_area"] #to find out the percentage of landcover on bauxite that is protected
landcover_bauxite_allprotected_classified = landcover_bauxite_allprotected.groupby('Classify').sum()

landcover_bauxite_allprotected_classified.to_csv(os.path.join(output_path, 'landcover_bauxite_allprotected_classified.csv'))

In [ ]:
print(landcover_bauxite_allprotected_classified)

In [ ]:
landcover_bauxite.plot()

In [ ]:
landcover_bauxite.groupby('Classify').sum()

In [ ]:
landcover_bauxite[['Hectares', 'Classify']].groupby('Classify').sum()

In [ ]:
all_protected_areas = geopandas.read_file(os.path.join(base_path, 'allprotectedareas.gpkg'))

In [ ]:
landcover_bauxite_allprotected = landcover_bauxite \
    .overlay(all_protected_areas, how='intersection')

In [ ]:
landcover_bauxite_allprotected.plot()

In [ ]:
landcover_bauxite_allprotected.to_file("landcover_bauxite_allprotected.shp")

In [ ]:
landcover_bauxite_allprotected = geopandas.read_file(os.path.join(base_path, 'landcover_bauxite_allprotected.gpkg'))

In [ ]:
landcover_bauxite_allprotected[['Hectares', 'Classify']].groupby('Classify').sum()